In [1]:
%load_ext autoreload
%autoreload 2

In [ ]:
from brain_image.utils import setup_logging


setup_logging()

In [ ]:
import logging
import pandas as pd
import json
import yaml
from pathlib import Path


def get_single_file(dir: Path, pattern: str) -> Path:
    paths = list(dir.rglob(pattern))

    num_results = len(paths)
    if num_results == 0:
        raise ValueError(f"Could not find any paths in dir {dir} matching pattern {pattern}")

    if num_results > 1:
        raise ValueError(f"Expected to find one results matching pattern {pattern} in dir {dir} - Found {num_results}: {tuple(paths)}")

    path = paths[0]
    return path
        

def gather_metrics(experiment_dir: Path, selected_hparams: list[str] = []) -> pd.DataFrame:
    all_metrics = []

    for exp_dir in experiment_dir.iterdir():
        metrics_path = get_single_file(exp_dir, "*test/metrics.json")

        logging.info(f"Loading metrics from {metrics_path}")

        with open(metrics_path, "r") as f:
            metrics = json.load(f)

        if len(selected_hparams) > 0:
            hparams_path = get_single_file(exp_dir, "*hparams.yaml")
            with open(hparams_path) as f:
                hparams = yaml.safe_load(f)

            for hparam_key in selected_hparams:
                hparam_parts = hparam_key.split(".")
                curr_hparam = hparams
                for part in hparam_parts:
                    curr_hparam = curr_hparam[part]

                metrics[hparam_key] = curr_hparam

        all_metrics.append(metrics)

    metrics = pd.DataFrame.from_records(all_metrics)
    return metrics


ex_path = Path("experiments/norm_scheme")
metrics = gather_metrics(ex_path, ["config.prior.norm_scheme"])
metrics

,prior/pixcorr,prior/ssim,prior/alex2,prior/alex5,prior/inceptionv3,prior/clip,prior/efficientnet,prior/swav,config.prior.norm_scheme
0,0.138323,0.272368,0.785427,0.823065,0.659171,0.682362,0.921153,0.618570,l2_scale
1,0.159358,0.298576,0.819296,0.874849,0.756633,0.831106,0.859854,0.550403,z_scale
2,0.152860,0.263793,0.804121,0.837915,0.700653,0.752990,0.897062,0.591040,none
